runpod/pytorch:2.2.1-py3.10-cuda12.1.1-devel-ubuntu22.04

pip uninstall -y transformers accelerate bitsandbytes peft trl

pip install transformers==4.45.2 accelerate==0.34.2 bitsandbytes==0.43.3 peft==0.12.0 trl==0.11.1

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

print("✅ 라이브러리 로드 완료!")

✅ 라이브러리 로드 완료!


In [2]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 
)

print("📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" 
)
print("✅ 모델 로드 성공!")

📦 모델 로딩 중... (약 15GB 데이터를 GPU로 올립니다)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ 모델 로드 성공!


In [5]:
# 메모리 절약을 위한 그래디언트 체크포인팅
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA 어댑터 설정
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

print("✅ LoRA 어댑터 장착 완료!")

✅ LoRA 어댑터 장착 완료!


In [6]:
# JSONL 파일 로드
dataset = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

# Qwen 대화 템플릿 입히기 + 기존 'messages' 컬럼 삭제 (이게 핵심입니다!)
dataset = dataset.map(
    lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False)},
    remove_columns=["messages"] # 에러의 주범인 messages 컬럼을 제거합니다.
)

# 토크나이저 패딩 토큰 설정 (Qwen의 경우 대개 설정되어 있지만 안전을 위해 추가)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 데이터 준비 완료! (남은 컬럼: {dataset['train'].column_names})")

✅ 데이터 준비 완료! (남은 컬럼: ['text'])


In [7]:
from transformers import TrainingArguments
from trl import SFTTrainer

# 안전을 위해 max_seq_length는 3072로 타협하거나, 
# 4096을 꼭 쓰셔야 한다면 eval_strategy="no"가 필수입니다.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=3072, # 4096에서 살짝 낮추면 로그 찍히는 속도가 훨씬 가벼워집니다.
    args=TrainingArguments(
        output_dir="./pypi_model_result",
        per_device_train_batch_size=1,      # 메모리 안전 제일!
        gradient_accumulation_steps=8,      # 8개 샘플마다 실제 업데이트 (학습 안정성)
        num_train_epochs=5,
        learning_rate=2e-4,
        
        # 로그 관련 설정 (가장 자주 찍히도록)
        logging_steps=1,                    # 매 스텝(8개 샘플 처리 시)마다 로그 출력
        logging_first_step=True,            # 시작하자마자 첫 로그 출력
        report_to="none",                   # 외부 툴 연동 없이 콘솔에만 집중
        
        # 메모리 및 성능 최적화
        bf16=True,                          # 4090의 축복, 수치 안정성 확보
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        
        # 평가 및 저장 (메모리 부족의 주범인 평가는 끕니다)
        eval_strategy="no",                 # 학습 중에 평가를 안 해야 OOM이 안 납니다.
        save_strategy="epoch",              # 대신 에폭마다 결과물은 꼬박꼬박 저장!
    ),
)

print("🔥 [초밀착 모니터링 모드] 학습을 시작합니다!")
trainer.train()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/226 [00:00<?, ? examples/s]

🔥 [초밀착 모니터링 모드] 학습을 시작합니다!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss
1,1.000500
2,0.758500
3,0.804800
4,0.723400
5,0.696800
6,0.816600
7,0.623800
8,0.684700
9,0.687200
10,0.646400


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-pac

TrainOutput(global_step=140, training_loss=0.3785802054618086, metrics={'train_runtime': 1883.6506, 'train_samples_per_second': 0.6, 'train_steps_per_second': 0.074, 'total_flos': 1.1798078452384358e+17, 'train_loss': 0.3785802054618086, 'epoch': 4.95575221238938})

In [11]:
import torch

# 1. 모델을 평가 모드로 전환 (학습 중단 및 추론 준비)
model.eval()

def test_ai_examiner(claim_text):
    # 2. 테스트용 메시지 구성 (학습 때와 동일한 형식)
    messages = [
        {"role": "system", "content": "당신은 대한민국 특허청의 베테랑 심사관입니다. 입력된 청구항의 기재불비 여부를 논리적으로 심사하여 답변하십시오."},
        {"role": "user", "content": f"다음 청구항을 심사하여 기재불비 사항이 있다면 지적해 주세요:\n\n{claim_text}"}
    ]
    
    # 3. 템플릿 적용 및 토크나이징
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # 4. 답변 생성 (추론)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,    # 약간의 창의성 (너무 낮으면 기계적, 너무 높으면 헛소리)
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 5. 결과 출력
    response = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    print("\n" + "="*50)
    print(f"🔎 [심사 대상 청구항]:\n{claim_text}")
    print("-" * 50)
    print(f"🤖 [AI 심사 결과]:\n{response}")
    print("="*50 + "\n")

# --- 여기서 테스트하고 싶은 청구항을 입력하세요! ---
test_claim = """청구항 1 \n하나 이상의 컴퓨터 및 명령어를 저장하는 하나 이상의 저장 장치를 포함하는 시스템으로서, 상기 명령어는 하\n나 이상의 컴퓨터에 의해 실행될 때, 하나 이상의 컴퓨터로 하여금 입력 시퀀스를 수신하고 그리고 상기 입력\n시퀀스를 프로세싱하여 출력을 생성하도록 구성된 어텐션 신경망을 구현하게 하며, 상기 어텐션 신경망은, \n어텐션 블록 입력으로부터 도출되는 쿼리 입력, 키 입력 및 값 입력을 수신하도록 구성된 어텐션 블록을 포함하\n며, 상기 어텐션 블록은,\n어텐션 신경망 계층 -상기 어텐션 신경망 계층은 \n 쿼리 입력, 키 입력 및 값 입력에서 도출된 어텐션 계층 입력을 수신하고, 그리고\n 어텐션 신경망 계층에 대한 어텐션 계층 출력을 생성하기 위해 어텐션 계층 입력에 어텐션 메커니즘을 적용하\n도록 구성됨-; 그리고\n게이팅 신경망 계층을 포함하며, 상기 게이팅 신경망 계층은 어텐션 신경망 계층의 어텐션 계층 출력 및 어텐션\n블록 입력에 게이팅 메커니즘을 적용하여 게이팅된 어텐션 출력을 생성하도록 구성되는 것을 특징으로 하는 시\n스템.\n청구항 2 \n제1항에 있어서, 상기 어텐션 블록은, \n계층 정규화 오퍼레이션을 쿼리 입력, 키 입력, 및 값 입력에 적용하여 정규화된 쿼리 입력, 정규화된 키 입력,\n및 정규화된 값 입력을 생성하도록 구성된 제1 계층 정규화 계층을 더 포함하고, \n상기 어텐션 계층 입력은 정규화된 쿼리 입력, 정규화된 키 입력, 및 정규화된 값 입력을 포함하는 것을 특징으\n로 하는 시스템.\n청구항 3 \n제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용\n하는 단계는,\n어텐션 블록 입력에 시그모이드 변조를 적용하여 제1 시그모이드 변조된 출력을 생성하는 단계; 그리고\n상기 제1 시그모이드 변조된 출력을 상기 어텐션 계층 출력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계\n를 포함하는 것을 특징으로 하는 시스템.\n청구항 4 \n제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용\n하는 단계는,\n어텐션 계층 출력에 시그모이드 변조를 적용하여 제2 시그모이드 변조된 출력을 생성하는 단계, 그리고\n제2 시그모이드 변조된 출력을 어텐션 블록 입력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계를 포함하는\n것을 특징으로 하는 시스템.\n청구항 5 \n제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용\n하는 단계는,\n시그모이드 가중치(weighting)를 사용하여 어텐션 블록 입력과 어텐션 계층 출력의 컨벡스(convex) 조합을 계산\n하여 게이팅된 어텐션 출력을 생성하는 단계를 포함하는 것을 특징으로 하는 시스템.\n청구항 6 \n제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용\n하는 단계는,\n어텐션 계층 출력에 시그모이드 및 하이퍼볼릭탄젠트(tanh) 활성화를 적용하여 시그모이드-tanh 출력을 생성하\n는 단계, 그리고\n상기 시그모이드-tanh 출력을 상기 어텐션 블록 입력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계를 포함\n하는 것을 특징으로 하는 시스템.\n청구항 7 \n제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용\n하는 단계는,\n상기 어텐션 블록 입력과 어텐션 계층 출력에 게이팅된 순환 유닛을 적용하는 단계를 포함하는 것을 특징으로\n하는 시스템.\n청구항 8 \n제1항 내지 제7항 중 어느 한 항에 있어서, 상기 어텐션 블록은,\n상기 게이팅된 어텐션 출력에 계층 정규화 오퍼레이션을 적용하여 정규화된-게이팅된 어텐션 출력을 생성하도록\n구성된 제2 계층 정규화 계층,\n정규화된-게이팅된 어텐션 출력에 하나 이상의 변환을 적용하여 템포러리(temporary) 어텐션 블록 출력을 생성\n하도록 구성된 하나 이상의 피드포워드 신경망 계층, 그리고\n제2 게이팅 메커니즘을 상기 템포러리 어텐션 블록 출력 및 상기 게이팅된 어텐션 출력에 적용하여 상기 어텐션\n블록에 대한 최종 어텐션 블록 출력을 생성하도록 구성된 제2 게이팅 신경망 계층을 더 포함하는 것을 특징으로\n하는 시스템.\n청구항 9 \n제1항 내지 제8항 중 어느 한 항에 있어서, 상기 어텐션 메커니즘은 셀프-어텐션(self-attention) 메커니즘인\n것을 특징으로 하는 시스템.\n청구항 10 \n제1항 내지 제8항 중 어느 한 항에 있어서, 상기 어텐션 메커니즘은 마스킹된 셀프-어텐션 메커니즘인 것을 특\n징으로 하는 시스템.\n청구항 11 \n하나 이상의 컴퓨터에 의해 실행될 때, 상기 하나 이상의 컴퓨터로 하여금 제1항 내지 제10항 중 어느 한 항의\n어텐션 신경망을 구현하게 하는 명령어를 저장하는 하나 이상의 컴퓨터 저장 매체. \n청구항 12 \n제1항 내지 제10항 중 어느 한 항의 어텐션 신경망이 수행하도록 구성된 동작을 포함하는 방법."""

test_ai_examiner(test_claim)


🔎 [심사 대상 청구항]:
청구항 1 
하나 이상의 컴퓨터 및 명령어를 저장하는 하나 이상의 저장 장치를 포함하는 시스템으로서, 상기 명령어는 하
나 이상의 컴퓨터에 의해 실행될 때, 하나 이상의 컴퓨터로 하여금 입력 시퀀스를 수신하고 그리고 상기 입력
시퀀스를 프로세싱하여 출력을 생성하도록 구성된 어텐션 신경망을 구현하게 하며, 상기 어텐션 신경망은, 
어텐션 블록 입력으로부터 도출되는 쿼리 입력, 키 입력 및 값 입력을 수신하도록 구성된 어텐션 블록을 포함하
며, 상기 어텐션 블록은,
어텐션 신경망 계층 -상기 어텐션 신경망 계층은 
 쿼리 입력, 키 입력 및 값 입력에서 도출된 어텐션 계층 입력을 수신하고, 그리고
 어텐션 신경망 계층에 대한 어텐션 계층 출력을 생성하기 위해 어텐션 계층 입력에 어텐션 메커니즘을 적용하
도록 구성됨-; 그리고
게이팅 신경망 계층을 포함하며, 상기 게이팅 신경망 계층은 어텐션 신경망 계층의 어텐션 계층 출력 및 어텐션
블록 입력에 게이팅 메커니즘을 적용하여 게이팅된 어텐션 출력을 생성하도록 구성되는 것을 특징으로 하는 시
스템.
청구항 2 
제1항에 있어서, 상기 어텐션 블록은, 
계층 정규화 오퍼레이션을 쿼리 입력, 키 입력, 및 값 입력에 적용하여 정규화된 쿼리 입력, 정규화된 키 입력,
및 정규화된 값 입력을 생성하도록 구성된 제1 계층 정규화 계층을 더 포함하고, 
상기 어텐션 계층 입력은 정규화된 쿼리 입력, 정규화된 키 입력, 및 정규화된 값 입력을 포함하는 것을 특징으
로 하는 시스템.
청구항 3 
제1항 또는 제2항에 있어서, 상기 어텐션 블록 입력 및 상기 어텐션 계층 출력에 대해 게이팅 메커니즘을 적용
하는 단계는,
어텐션 블록 입력에 시그모이드 변조를 적용하여 제1 시그모이드 변조된 출력을 생성하는 단계; 그리고
상기 제1 시그모이드 변조된 출력을 상기 어텐션 계층 출력과 결합하여 게이팅된 어텐션 출력을 생성하는 단계
를 포함하는 것을 특징으로 하는 시스템.
청구항 4 
제1항 또는 제2항에 있어서, 

In [12]:
import torch

# 1. 모델을 평가 모드로 전환 (학습 중단 및 추론 준비)
model.eval()

def test_ai_examiner(claim_text):
    # 2. 테스트용 메시지 구성 (학습 때와 동일한 형식)
    messages = [
        {"role": "system", "content": "당신은 대한민국 특허청의 베테랑 심사관입니다. 입력된 청구항의 기재불비 여부를 논리적으로 심사하여 답변하십시오."},
        {"role": "user", "content": f"다음 청구항을 심사하여 기재불비 사항이 있다면 지적해 주세요:\n\n{claim_text}"}
    ]
    
    # 3. 템플릿 적용 및 토크나이징
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # 4. 답변 생성 (추론)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,    # 약간의 창의성 (너무 낮으면 기계적, 너무 높으면 헛소리)
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 5. 결과 출력
    response = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    print("\n" + "="*50)
    print(f"🔎 [심사 대상 청구항]:\n{claim_text}")
    print("-" * 50)
    print(f"🤖 [AI 심사 결과]:\n{response}")
    print("="*50 + "\n")

# --- 여기서 테스트하고 싶은 청구항을 입력하세요! ---
test_claim = """청구항 1 \n인공지능을 이용한 미술 자산 가치 평가 시스템에 있어서,\n인스트럭션들을 저장하는 메모리,\n프로세서를 포함하고,\n상기 인스트럭션들은, 상기 프로세서에 의해 실행 시에, 상기 시스템이\n복수의 데이터 소스로부터 미술 작품 관련 데이터를 수집하고,\n수집된 데이터를 이용하여 가치 평가 모델을 학습하며,\n학습된 모델을 이용하여 미술 작품의 가치를 평가하고,\n평가 결과를 생성하도록 제어하고,\n상기 프로세서에 의해 실행 시에, 상기 시스템이\n학습 데이터셋의 각 샘플에 대해 초기 가중치를 할당하여 제1 가중치 벡터를 생성하고, 이를 이용하여 제1 가중\n치 처리 샘플을 생성하여 저장하며,\n정보 이득율 기준의 최적 분할 특징을 선택하여 제1 의사결정 트리 모델을 학습시키고, 예측값과 실제 가치의\n차이를 계산하여 제1 전체 오차를 산출하고,\n산출된 예측 오차를 기반으로 제2 가중치 벡터를 생성하고, 이를 이용하여 제2 의사결정 트리 모델을 학습시켜\n제2 전체 오차를 산출하며,\n오차 감소율이 임계값 이하가 될 때까지 반복 학습을 수행하여 복수의 의사결정 트리 모델과 각각의 전체 오차\n를 획득하고,\n각 모델의 오차를 기반으로 기여도 계수를 산출하여 의사결정 트리 모델들을 선형 결합하며,\n예측 안정성 향상을 위해 지수 이동 평균을 적용하고 신뢰구간을 계산하여,\n평가 대상 미술 작품에 대한 최종 가치 평가값과 신뢰구간을 산출하도록 제어하는 미술 자산 가치 평가 시스템.\n청구항 2 \n제 1항에 있어서,\n상기 미술 작품 관련 데이터는\n작품의 물리적 특성 데이터, 작가의 경력 데이터 및 시장 거래 데이터를 포함하는 미술 자산 가치 평가 시스템.\n청구항 3 \n제 1항에 있어서,\n상기 가치 평가 모델은\n현재 가치 예측 모델, 가격 변동 예측 모델 및 투자 가치 평가 모델을 포함하는 미술 자산 가치 평가 시스템.\n청구항 4 \n제 1항에 있어서,\n상기 시스템은 \n작품의 희소성 지표, 성장성 지표, 시장 인지도 지표 및 컬렉터 관심도 지표를 산출하여 투자 가치를 평가하는\n미술 자산 가치 평가 시스템.\n청구항 5 \n제 1항에 있어서,\n상기 시스템은 각 평가 모델의 신뢰도 지수를 생성하고,\n신뢰도 지수에 기반하여 가중치를 적용한 통합 평가 점수를 산출하는 미술 자산 가치 평가 시스템.\n청구항 6 \n제 1항에 있어서,\n상기 시스템은 작품의 이미지를 분석하여\n색상 평가 지수, 시각적 효과 지수, 기술적 완성도 지수 및 공간감 지수를 산출하는 미술 자산 가치 평가 시스\n템.\n청구항 7 \n제 1항에 있어서,\n상기 시스템은 작품의 2차원 이미지로부터 3차원 모델을 생성하고,\n생성된 3차원 모델을 기반으로 작품의 특징을 분석하여 평가하는 미술 자산 가치 평가 시스템.\n청구항 8 \n제 1항에 있어서,\n상기 시스템은 도메인 적응 기법을 이용하여\n소스 도메인의 데이터를 타겟 도메인으로 전이하고,\n전이된 데이터를 이용하여 가치 평가 모델을 학습하는 미술 자산 가치 평가 시스템.\n청구항 9 \n제 1항에 있어서,\n상기 시스템은 시계열 데이터베이스를 구축하고,\n시장 변동 패턴, 감성 분석 결과 및 품질 지수를 기반으로 미술품의 가치 변동을 예측하는 미술 자산 가치 평가\n시스템."""

test_ai_examiner(test_claim)


🔎 [심사 대상 청구항]:
청구항 1 
인공지능을 이용한 미술 자산 가치 평가 시스템에 있어서,
인스트럭션들을 저장하는 메모리,
프로세서를 포함하고,
상기 인스트럭션들은, 상기 프로세서에 의해 실행 시에, 상기 시스템이
복수의 데이터 소스로부터 미술 작품 관련 데이터를 수집하고,
수집된 데이터를 이용하여 가치 평가 모델을 학습하며,
학습된 모델을 이용하여 미술 작품의 가치를 평가하고,
평가 결과를 생성하도록 제어하고,
상기 프로세서에 의해 실행 시에, 상기 시스템이
학습 데이터셋의 각 샘플에 대해 초기 가중치를 할당하여 제1 가중치 벡터를 생성하고, 이를 이용하여 제1 가중
치 처리 샘플을 생성하여 저장하며,
정보 이득율 기준의 최적 분할 특징을 선택하여 제1 의사결정 트리 모델을 학습시키고, 예측값과 실제 가치의
차이를 계산하여 제1 전체 오차를 산출하고,
산출된 예측 오차를 기반으로 제2 가중치 벡터를 생성하고, 이를 이용하여 제2 의사결정 트리 모델을 학습시켜
제2 전체 오차를 산출하며,
오차 감소율이 임계값 이하가 될 때까지 반복 학습을 수행하여 복수의 의사결정 트리 모델과 각각의 전체 오차
를 획득하고,
각 모델의 오차를 기반으로 기여도 계수를 산출하여 의사결정 트리 모델들을 선형 결합하며,
예측 안정성 향상을 위해 지수 이동 평균을 적용하고 신뢰구간을 계산하여,
평가 대상 미술 작품에 대한 최종 가치 평가값과 신뢰구간을 산출하도록 제어하는 미술 자산 가치 평가 시스템.
청구항 2 
제 1항에 있어서,
상기 미술 작품 관련 데이터는
작품의 물리적 특성 데이터, 작가의 경력 데이터 및 시장 거래 데이터를 포함하는 미술 자산 가치 평가 시스템.
청구항 3 
제 1항에 있어서,
상기 가치 평가 모델은
현재 가치 예측 모델, 가격 변동 예측 모델 및 투자 가치 평가 모델을 포함하는 미술 자산 가치 평가 시스템.
청구항 4 
제 1항에 있어서,
상기 시스템은 
작품의 희소성 지표, 성장성 지표, 시장 인지도 지표 및 컬렉터 관심도 지표를 산출하여 투자 가치를

In [13]:
# 1. 저장 경로 설정
save_path = "pypi_patent_examiner_lora"

# 2. 모델(어댑터 가중치) 및 토크나이저 저장
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"🎉 성공! 학습된 모델이 '{save_path}' 폴더에 저장되었습니다.")
print("이제 이 폴더만 있으면 언제든 AI 심사관을 소환할 수 있습니다.")

🎉 성공! 학습된 모델이 'pypi_patent_examiner_lora' 폴더에 저장되었습니다.
이제 이 폴더만 있으면 언제든 AI 심사관을 소환할 수 있습니다.


In [14]:
import shutil

# 학습된 폴더를 pypi_model.zip 이라는 이름으로 압축합니다.
shutil.make_archive("pypi_model", 'zip', "pypi_patent_examiner_lora")

print("✅ 압축 완료! 이제 왼쪽 파일 탐색기에서 'pypi_model.zip'을 찾아 다운로드하세요.")

✅ 압축 완료! 이제 왼쪽 파일 탐색기에서 'pypi_model.zip'을 찾아 다운로드하세요.
